# 00b – Linear algebra review (NumPy)

In [1]:
# Basis states and unit norm checks
import numpy as np

ket0 = np.array([1.0, 0.0], dtype=complex)
ket1 = np.array([0.0, 1.0], dtype=complex)
ket_plus = (ket0 + ket1) / np.sqrt(2)
ket_minus = (ket0 - ket1) / np.sqrt(2)

for name, state in [("|0>", ket0), ("|1>", ket1), ("|+>", ket_plus), ("|->", ket_minus)]:
    norm = np.linalg.norm(state)
    print(f"{name:4s}  norm={norm:.6f}  vec={state.real}")
    assert np.isclose(norm, 1.0)

|0>   norm=1.000000  vec=[1. 0.]
|1>   norm=1.000000  vec=[0. 1.]
|+>   norm=1.000000  vec=[0.70710678 0.70710678]
|->   norm=1.000000  vec=[ 0.70710678 -0.70710678]


In [2]:
# Inner products (<0|1> = 0, <0|+> = 1/sqrt(2)) and projectors P0 + P1 = I
inner_01 = ket0.conj().T @ ket1
inner_0p = ket0.conj().T @ ket_plus
print("<0|1> =", round(float(inner_01.real), 6))
print("<0|+> =", round(float(inner_0p.real), 6))
assert np.isclose(inner_01, 0.0)
assert np.isclose(inner_0p, 1.0 / np.sqrt(2))

P0 = np.outer(ket0, ket0.conj())
P1 = np.outer(ket1, ket1.conj())
I2 = np.eye(2, dtype=complex)
assert np.allclose(P0 + P1, I2)
print("P0 + P1 == I2 verified")

<0|1> = 0.0
<0|+> = 0.707107
P0 + P1 == I2 verified


In [3]:
# Unitary (U^dagger U = I) vs. Hermitian (M^dagger = M) checks on single-qubit gates
def is_unitary(U):
    return np.allclose(U.conj().T @ U, np.eye(U.shape[0], dtype=complex))

def is_hermitian(M):
    return np.allclose(M.conj().T, M)

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)
Ry_pi3 = np.array([
    [np.cos(np.pi / 6), -np.sin(np.pi / 6)],
    [np.sin(np.pi / 6),  np.cos(np.pi / 6)],
], dtype=complex)

for name, mat in [("X", X), ("Y", Y), ("Z", Z), ("H", H), ("S", S), ("T", T), ("Ry(pi/3)", Ry_pi3)]:
    u, h = is_unitary(mat), is_hermitian(mat)
    print(f"{name:10s}  unitary={u}  hermitian={h}")
    assert u
    assert h == (name in {"X", "Y", "Z", "H"})

X           unitary=True  hermitian=True
Y           unitary=True  hermitian=True
Z           unitary=True  hermitian=True
H           unitary=True  hermitian=True
S           unitary=True  hermitian=False
T           unitary=True  hermitian=False
Ry(pi/3)    unitary=True  hermitian=False


In [4]:
# Kronecker products for 2-qubit basis states and operators (H x I, Z x Z)
basis_2q = {
    "|00>": np.kron(ket0, ket0),
    "|01>": np.kron(ket0, ket1),
    "|10>": np.kron(ket1, ket0),
    "|11>": np.kron(ket1, ket1),
}
for name, vec in basis_2q.items():
    assert np.isclose(np.linalg.norm(vec), 1.0)

H_I = np.kron(H, I2)
ZZ = np.kron(Z, Z)
print("diag(Z x Z):", np.diag(ZZ).real.astype(int))
assert np.allclose(np.diag(ZZ), [1, -1, -1, 1])
assert is_unitary(H_I) and is_unitary(ZZ)

diag(Z x Z): [ 1 -1 -1  1]


## Single-edge Max-Cut operator
With $z_i = 1 - 2x_i \in \{+1, -1\}$, a single edge $(i,j)$ has cost $C_{ij} = \frac{1}{2}(I_4 - Z \otimes Z)$: $0$ for $|00\rangle, |11\rangle$ and $1$ for $|01\rangle, |10\rangle$.

In [5]:
# Single-edge Max-Cut cost operator: C_ij = 0.5 * (I4 - Z x Z)
I4 = np.eye(4, dtype=complex)
C_ij = 0.5 * (I4 - ZZ)
print("diag(C_ij):", np.diag(C_ij).real)
assert np.allclose(np.diag(C_ij).real, [0.0, 1.0, 1.0, 0.0])

for name, state in basis_2q.items():
    val = float((state.conj().T @ C_ij @ state).real)
    print(f"<{name[1:-1]}| C_ij |{name[1:-1]}> = {val:.1f}")
    assert np.isclose(val, 0.0 if name in {"|00>", "|11>"} else 1.0)

diag(C_ij): [0. 1. 1. 0.]
<00| C_ij |00> = 0.0
<01| C_ij |01> = 1.0
<10| C_ij |10> = 1.0
<11| C_ij |11> = 0.0
